# 64 - Early Fusion on Primer conf60

**Arsitektur:** Early Fusion — landmark heatmap (Gaussian 224×224) di-stack sebagai channel ke-4 pada citra RGB → input 4-channel CNN.

**Referensi:** Wu et al. (MMM 2020) — *Emotion Recognition with Facial Landmark Heatmaps* (HAE-Net).

**Dataset:** Primer conf60 (6,795 samples, 37 users, confidence ≥ 60%).
**Backbone:** Scratch (`EmotionEarlyFusion`) + Transfer Learning (`EmotionEarlyFusionTransfer` dengan ResNet18).
**Skenario:**
- **B1** Baseline (no class weights, base train set)
- **B2** Class Weights (weighted CE loss, base train set)
- **B3** Weights + Augmentation (weighted CE + augmented train set `dataset_frontonly_conf60_augmented`)

**Kelas:** 7-class + 4-class remap.
**Metrik:** Macro / Micro / Weighted F1.

**Total konfigurasi:** 2 backbone × 3 skenario × 2 kelas = **12 configs**.

**Prerequisite**:
1. Base heatmap: `python scripts/generate_landmark_heatmaps.py --only "Primer conf60"`
2. Augmented heatmap (untuk B3): `python scripts/generate_landmark_heatmaps.py --only "augmented"`
3. `data/dataset_frontonly_conf60_augmented/` harus sudah ada (dipakai juga oleh nb 43-57)

Note: B3 akan di-skip otomatis kalau augmented dataset atau heatmap-nya tidak ada.

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import EmotionEarlyFusion, EmotionEarlyFusionTransfer
from training.utils import train_model, full_evaluation, get_class_weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15
OUTPUT_DIR = PROJECT_ROOT / 'models' / 'frontonly_conf60' / 'early_fusion'
os.makedirs(OUTPUT_DIR, exist_ok=True)

REMAP_4 = np.array([0, 1, 2, 3, 3, 3, 3], dtype=np.int64)

print('Setup complete.')

Device: cuda
GPU: Tesla T4
Setup complete.


In [2]:
# ── Data loading: stack RGB image + landmark heatmap -> 4-channel ──

PRIMER_DIR = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60'
PRIMER_AUG_DIR = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60_augmented'


def load_split(split, root=None):
    """Load image + heatmap, return (N, 224, 224, 4) float32 + labels."""
    root = root or PRIMER_DIR
    img = np.load(root / f'X_{split}_images.npy')       # (N, 224, 224, 3)
    heat = np.load(root / f'X_{split}_heatmaps.npy')    # (N, 224, 224)
    y = np.load(root / f'y_{split}.npy')
    if heat.ndim == 3:
        heat = heat[..., None]  # (N, 224, 224, 1)
    x4 = np.concatenate([img, heat], axis=-1)  # (N, 224, 224, 4)
    return x4.astype(np.float32, copy=False), y


# Base (B1, B2)
X_tr, y_tr_7 = load_split('train')
X_v,  y_v_7  = load_split('val')
X_te, y_te_7 = load_split('test')

# Augmented train (B3 only) — uses balanced/augmented train set
# Val & test stay the same (from base)
if PRIMER_AUG_DIR.exists() and (PRIMER_AUG_DIR / 'X_train_heatmaps.npy').exists():
    X_tr_aug, y_tr_aug_7 = load_split('train', root=PRIMER_AUG_DIR)
    print(f'Train aug 4-ch: {X_tr_aug.shape} (for B3)')
    HAS_AUG = True
else:
    print('[WARN] augmented dataset or its heatmap missing; B3 will be skipped')
    X_tr_aug, y_tr_aug_7 = None, None
    HAS_AUG = False

y_tr_4 = REMAP_4[y_tr_7]
y_v_4 = REMAP_4[y_v_7]
y_te_4 = REMAP_4[y_te_7]
y_tr_aug_4 = REMAP_4[y_tr_aug_7] if HAS_AUG else None

print(f'Train base 4-ch: {X_tr.shape}  Val: {X_v.shape}  Test: {X_te.shape}')
print(f'Train 7-class dist: {Counter(y_tr_7.tolist())}')
print(f'Train 4-class dist: {Counter(y_tr_4.tolist())}')
if HAS_AUG:
    print(f'Train aug 7-class dist: {Counter(y_tr_aug_7.tolist())}')

Train aug 4-ch: (5829, 224, 224, 4) (for B3)
Train base 4-ch: (5287, 224, 224, 4)  Val: (579, 224, 224, 4)  Test: (929, 224, 224, 4)
Train 7-class dist: Counter({0: 4526, 1: 416, 2: 287, 3: 27, 6: 16, 5: 13, 4: 2})
Train 4-class dist: Counter({0: 4526, 1: 416, 2: 287, 3: 58})
Train aug 7-class dist: Counter({0: 4526, 1: 416, 2: 287, 3: 150, 5: 150, 6: 150, 4: 150})


In [3]:
# ── Helpers ──

def make_loader(x4, y, batch_size=32, shuffle=True):
    # (N, 224, 224, 4) -> (N, 4, 224, 224)
    t = torch.from_numpy(x4).permute(0, 3, 1, 2).contiguous()
    ys = torch.from_numpy(y).long()
    ds = TensorDataset(t, ys)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)


def train_and_eval(model_class, lr, tr_x, tr_y, v_x, v_y, te_x, te_y,
                   emotions, save_dir, class_weights=None):
    tr_loader = make_loader(tr_x, tr_y, BATCH_SIZE)
    val_loader = make_loader(v_x, v_y, BATCH_SIZE, False)
    test_loader = make_loader(te_x, te_y, BATCH_SIZE, False)

    model = model_class(num_classes=len(emotions)).to(device)
    save_path = str(save_dir / 'model.pth')

    criterion = nn.CrossEntropyLoss(weight=class_weights) if class_weights is not None \
                else nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=8, min_lr=1e-7)

    train_model(model, tr_loader, val_loader, criterion, optimizer, scheduler,
                device, 'cnn', EPOCHS, PATIENCE, save_path)
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    r = full_evaluation(model, test_loader, criterion, device, 'cnn', emotions)
    return {
        'accuracy': float(r['test_accuracy']),
        'macro_f1': float(r['test_macro_f1']),
        'micro_f1': float(r['test_micro_f1']),
        'weighted_f1': float(r['test_weighted_f1']),
    }


def compute_class_weights(y, num_classes):
    """Inverse-frequency class weights."""
    counts = np.bincount(y, minlength=num_classes).astype(np.float32)
    counts = np.where(counts == 0, 1.0, counts)
    w = (1.0 / counts)
    w = w * num_classes / w.sum()
    return torch.from_numpy(w).to(device)


def run_early_fusion(num_classes, emotions,
                      tr_x, tr_y, v_x, v_y, te_x, te_y,
                      tr_x_aug=None, tr_y_aug=None):
    """Run 6 configs per class: EarlyFusion(scratch/TL) × (B1, B2, B3).
    B3 uses augmented train data (if provided).
    """
    print(f"\n{'='*70}")
    print(f'  EARLY FUSION — Primer conf60 ({num_classes}-class)')
    print(f"{'='*70}")
    results = {}
    cw = compute_class_weights(tr_y, num_classes)
    cw_aug = compute_class_weights(tr_y_aug, num_classes) if tr_y_aug is not None else None

    # (key, model_class, lr, class_weights, use_augmented)
    configs = [
        ('EarlyFusion_B1',    EmotionEarlyFusion,         0.0001, None, False),
        ('EarlyFusion_B2',    EmotionEarlyFusion,         0.0001, cw,   False),
        ('EarlyFusion_B3',    EmotionEarlyFusion,         0.0001, cw_aug, True),
        ('EarlyFusion_TL_B1', EmotionEarlyFusionTransfer, 0.00005, None, False),
        ('EarlyFusion_TL_B2', EmotionEarlyFusionTransfer, 0.00005, cw,   False),
        ('EarlyFusion_TL_B3', EmotionEarlyFusionTransfer, 0.00005, cw_aug, True),
    ]

    for key, model_cls, lr, class_weights, use_aug in configs:
        if use_aug and tr_x_aug is None:
            print(f'\n  >> {key} SKIPPED (no augmented data)')
            continue
        print(f'\n  >> {key} ...')
        save_dir = OUTPUT_DIR / f'{num_classes}c' / key
        os.makedirs(save_dir, exist_ok=True)
        train_x = tr_x_aug if use_aug else tr_x
        train_y = tr_y_aug if use_aug else tr_y
        r = train_and_eval(model_cls, lr,
                           train_x, train_y, v_x, v_y, te_x, te_y,
                           emotions, save_dir,
                           class_weights=class_weights)
        results[key] = r
        print(f"    Macro={r['macro_f1']:.4f}  Micro={r['micro_f1']:.4f}  Weighted={r['weighted_f1']:.4f}")

    save_path = OUTPUT_DIR / f'early_fusion_{num_classes}c_results.json'
    with open(save_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'\n  Saved: {save_path}')
    return results

print('Helpers ready.')

Helpers ready.


## Run Early Fusion Experiments

In [4]:
EMOTIONS_7 = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgusted', 'surprised']
EMOTIONS_4 = ['neutral', 'happy', 'sad', 'negative']

res_7c = run_early_fusion(7, EMOTIONS_7,
                            X_tr, y_tr_7, X_v, y_v_7, X_te, y_te_7,
                            tr_x_aug=X_tr_aug, tr_y_aug=y_tr_aug_7)
res_4c = run_early_fusion(4, EMOTIONS_4,
                            X_tr, y_tr_4, X_v, y_v_4, X_te, y_te_4,
                            tr_x_aug=X_tr_aug, tr_y_aug=y_tr_aug_4)


  EARLY FUSION — Primer conf60 (7-class)



  >> EarlyFusion_B1 ...


 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      1.2998     0.6092     0.9725    0.8187   0.1381   0.000100  (34.9s)


     2      0.6258     0.8597     0.7745    0.8204   0.1496   0.000100  (34.6s)


...

    15      0.0205     0.9983     1.8144    0.8031   0.3443   0.000025  (19.5s)


    16      0.0200     0.9979     1.9515    0.8169   0.3385   0.000025  (19.5s)


    17      0.0295     0.9950     2.0228    0.8204   0.3182   0.000025  (19.4s)

Early stopping at epoch 17. Best epoch: 2 (val_f1=0.4527)

Best: epoch 2, val_acc=0.7789, val_f1=0.4527
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/frontonly_conf60/early_fusion/4c/EarlyFusion_TL_B3/model.pth


Test Loss: 0.7987
Test Accuracy: 0.6781
Test Macro F1:    0.4328
Test Micro F1:    0.6781
Test Weighted F1: 0.7086

Classification Report:
              precision    recall  f1-score   support

     neutral       0.99      0.64      0.77       688
       happy       0.44      0.92      0.59       183
         sad       0.30      0.48      0.37        50
    negative       0.00      0.00      0.00         8

    accuracy                           0.68       929
   macro avg       0.43      0.51      0.43       929
weighted avg       0.83      0.68      0.71       929



    Macro=0.4328  Micro=0.6781  Weighted=0.7086

  Saved: /home/bs000716/MOTHER-TANK/TRAIN/models/frontonly_conf60/early_fusion/early_fusion_4c_results.json


## Ringkasan Early Fusion (Semua Metrik)

In [5]:
for nc_label, res in [('7-class', res_7c), ('4-class', res_4c)]:
    print(f"\n{'='*78}")
    print(f'  Early Fusion on Primer conf60 {nc_label} — sorted by Macro F1')
    print(f"{'='*78}")
    print(f"  {'Config':<22} {'Macro':>10} {'Micro':>10} {'Weighted':>10} {'Accuracy':>10}")
    print(f"  {'-'*70}")
    for key in sorted(res.keys(), key=lambda k: -res[k]['macro_f1']):
        r = res[key]
        print(f"  {key:<22} {r['macro_f1']:>10.4f} {r['micro_f1']:>10.4f} {r['weighted_f1']:>10.4f} {r['accuracy']:>10.4f}")


  Early Fusion on Primer conf60 7-class — sorted by Macro F1
  Config                      Macro      Micro   Weighted   Accuracy
  ----------------------------------------------------------------------
  EarlyFusion_TL_B3          0.3330     0.7535     0.7729     0.7535
  EarlyFusion_B3             0.2638     0.6803     0.7260     0.6803
  EarlyFusion_TL_B1          0.2533     0.7126     0.7224     0.7126
  EarlyFusion_TL_B2          0.2467     0.6362     0.6634     0.6362
  EarlyFusion_B1             0.2464     0.7944     0.7861     0.7944
  EarlyFusion_B2             0.2049     0.5199     0.5516     0.5199

  Early Fusion on Primer conf60 4-class — sorted by Macro F1
  Config                      Macro      Micro   Weighted   Accuracy
  ----------------------------------------------------------------------
  EarlyFusion_TL_B1          0.4710     0.7696     0.7697     0.7696
  EarlyFusion_B1             0.4574     0.8224     0.8160     0.8224
  EarlyFusion_TL_B3          0.4328     